# 回測結果分析
分析各 Layer 的交易結果。修改下方 `LAYER` 切換要分析的版本。

In [ ]:
%%time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc('font', family='Microsoft JhengHei')

# ── 改這裡切換 Layer ──────────────────────────────────
LAYER = 'layer1'   # 'layer1' / 'layer2' / 'layer3' / ...
# ──────────────────────────────────────────────────────

TRADE_CSV  = f'output/{LAYER}_trades.csv'
EQUITY_CSV = f'output/{LAYER}_equity.csv'
TAIEX_CSV  = '../market_data/taiex.csv'
STOCKS_DIR = '../stocks_full'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.read_csv(TRADE_CSV, parse_dates=['entry_date', 'exit_date'])
df['hold_days'] = (df['exit_date'] - df['entry_date']).dt.days
df['pnl_net']   = df['pnl']

print(f'[{LAYER}] 總交易筆數：{len(df)}')
df.head()

## 1. 出場原因分析

In [ ]:
%%time
summary = df.groupby('exit_reason').agg(
    筆數=('pnl_net', 'count'),
    勝率=('pnl_net', lambda x: (x > 0).mean()),
    平均損益=('pnl_net', 'mean'),
    總損益=('pnl_net', 'sum'),
).round(1)
summary['勝率'] = summary['勝率'].map('{:.1%}'.format)
print(summary.to_string())

## 2. 持有天數分佈

In [ ]:
%%time
print(df['hold_days'].describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
df['hold_days'].hist(bins=40, ax=ax, color='#1565C0', alpha=0.7)
ax.set_title(f'[{LAYER}] 持有天數分佈')
ax.set_xlabel('天數')
ax.set_ylabel('筆數')
plt.tight_layout()
plt.show()

## 3. 年度損益分析

In [ ]:
%%time
df['year'] = df['exit_date'].dt.year
yearly = df.groupby('year').agg(
    筆數=('pnl_net', 'count'),
    總損益=('pnl_net', 'sum'),
    勝率=('pnl_net', lambda x: (x > 0).mean()),
).round(1)
yearly['勝率'] = yearly['勝率'].map('{:.1%}'.format)
print(yearly.to_string())

df.groupby('year')['pnl_net'].sum().plot(
    kind='bar', figsize=(12, 4), color='#1565C0', alpha=0.8
)
plt.axhline(0, color='red', linewidth=0.8)
plt.title(f'[{LAYER}] 年度總損益（元）')
plt.ylabel('損益')
plt.tight_layout()
plt.show()

## 4. TAIEX 大盤環境 vs 交易結果

In [ ]:
%%time
from pathlib import Path

if not Path(TAIEX_CSV).exists():
    print(f'[警告] 找不到 {TAIEX_CSV}，略過大盤環境分析')
else:
    taiex = pd.read_csv(TAIEX_CSV, index_col=0, parse_dates=True)
    close_col = 'Close' if 'Close' in taiex.columns else taiex.columns[3]
    taiex['EMA_200'] = taiex[close_col].ewm(span=200, adjust=False).mean()
    taiex['regime']  = (taiex[close_col] > taiex['EMA_200']).map({True: '多頭', False: '空頭'})
    regime_map   = taiex['regime'].to_dict()
    df['regime'] = df['entry_date'].map(regime_map)

    print('大盤環境分佈（進場日）：')
    print(df['regime'].value_counts())
    print()
    for r, g in df.groupby('regime'):
        wins = g[g.pnl_net > 0].pnl_net.sum()
        loss = g[g.pnl_net < 0].pnl_net.abs().sum()
        pf   = wins / loss if loss > 0 else float('inf')
        print(f"regime={r}  筆數={len(g):4d}  勝率={(g.pnl_net>0).mean():.1%}  "
              f"PF={pf:.3f}  總損益={g.pnl_net.sum():.0f}")

## 5. 贏家 vs 輸家 持有天數

In [ ]:
%%time
wins_df   = df[df.pnl_net > 0]
losses_df = df[df.pnl_net <= 0]

print(f"贏家平均持有天數：{wins_df.hold_days.mean():.1f} 天  中位數：{wins_df.hold_days.median():.0f} 天")
print(f"輸家平均持有天數：{losses_df.hold_days.mean():.1f} 天  中位數：{losses_df.hold_days.median():.0f} 天")
print()
print("結論：若贏家持有天數 < 輸家，代表贏的太早出場、輸的拖太久")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
wins_df.hold_days.hist(bins=30, ax=axes[0], color='#2E7D32', alpha=0.7)
axes[0].set_title('贏家持有天數')
axes[0].set_xlabel('天數')
losses_df.hold_days.hist(bins=30, ax=axes[1], color='#C62828', alpha=0.7)
axes[1].set_title('輸家持有天數')
axes[1].set_xlabel('天數')
plt.suptitle(f'[{LAYER}] 贏家 vs 輸家 持有天數')
plt.tight_layout()
plt.show()

## 6. 出場原因詳細分析

In [ ]:
%%time
for reason, g in df.groupby('exit_reason'):
    wins = g[g.pnl_net > 0].pnl_net.sum()
    loss = g[g.pnl_net < 0].pnl_net.abs().sum()
    pf   = wins / loss if loss > 0 else float('inf')
    print(f"{reason:15s}  筆數={len(g):4d} ({len(g)/len(df):.0%})  "
          f"勝率={(g.pnl_net>0).mean():.1%}  "
          f"平均持倉={g.hold_days.mean():.1f}天  "
          f"PF={pf:.3f}  總損益={g.pnl_net.sum():.0f}")

## 7. 最大虧損前 20 筆

In [ ]:
%%time
cols = ['ticker', 'entry_date', 'exit_date', 'hold_days',
        'entry_price', 'exit_price', 'pnl_net', 'exit_reason']
worst = df.nsmallest(20, 'pnl_net')[cols]
print(worst.to_string(index=False))

## 8. 持有超過 30 天但損益 <= 0（白持的交易）

In [ ]:
%%time
cols = ['ticker', 'entry_date', 'exit_date', 'hold_days',
        'entry_price', 'exit_price', 'pnl_net', 'exit_reason']
long_losers = df[(df.hold_days >= 30) & (df.pnl_net <= 0)].sort_values('pnl_net')
print(f"持有 >= 30 天且虧損：{len(long_losers)} 筆  總虧損：{long_losers.pnl_net.sum():.0f} 元")
print()
print(long_losers[cols].to_string(index=False))

## 9. 個股走勢分析

**操作說明：**
1. 修改 `trades_to_analyze` 清單，填入 `(股票代號, 進場日期)`
2. 留空則自動取最大獲利 & 最大虧損各 3 筆

In [ ]:
%%time
import os

def plot_trade(ticker, entry_date, df, window=30):
    trade = df[
        (df.ticker.astype(str) == str(ticker)) &
        (df.entry_date == pd.Timestamp(entry_date))
    ]
    if trade.empty:
        print(f'找不到 {ticker} {entry_date} 的交易紀錄')
        return
    trade = trade.iloc[0]

    csv_path = os.path.join(STOCKS_DIR, f'{ticker}.csv')
    if not os.path.exists(csv_path):
        print(f'找不到原始股票資料：{csv_path}')
        return

    stock = pd.read_csv(csv_path, encoding='utf-8-sig', parse_dates=['日期'])
    stock = stock.rename(columns={
        '日期': 'Date', '開盤價': 'Open', '最高價': 'High',
        '最低價': 'Low', '收盤價': 'Close', '成交股數': 'Volume'
    })
    for col in ['Open', 'High', 'Low', 'Close']:
        stock[col] = pd.to_numeric(stock[col], errors='coerce')
    stock = stock.set_index('Date').sort_index()

    entry_dt = pd.Timestamp(entry_date)
    exit_dt  = trade.exit_date
    idx      = stock.index
    ep       = idx.searchsorted(entry_dt)
    xp       = idx.searchsorted(exit_dt)
    sub      = stock.iloc[max(0, ep - window): min(len(idx), xp + window + 1)]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 7),
        gridspec_kw={'height_ratios': [3, 1]}, sharex=True
    )
    ax1.plot(sub.index, sub['Close'], color='#1565C0', linewidth=1.5, label='收盤價')
    ax1.fill_between(sub.index, sub['Low'], sub['High'], alpha=0.12, color='#1565C0', label='當日高低')
    ax1.axvline(entry_dt, color='green', linewidth=1.5, linestyle='--', alpha=0.8)
    ax1.axvline(exit_dt,  color='red',   linewidth=1.5, linestyle='--', alpha=0.8)
    ax1.scatter([entry_dt], [trade.entry_price], color='green', s=120, zorder=5,
                label=f'進場 {trade.entry_price:.1f}')
    ax1.scatter([exit_dt], [trade.exit_price], color='red', s=120, zorder=5, marker='v',
                label=f'出場 {trade.exit_price:.1f}')
    pnl_color = '#2E7D32' if trade.pnl_net >= 0 else '#C62828'
    ax1.set_title(
        f'[{LAYER}] {ticker}  進場: {entry_dt.date()} @ {trade.entry_price:.1f}  '
        f'出場: {exit_dt.date()} @ {trade.exit_price:.1f}  '
        f'持倉: {trade.hold_days}天  損益: {trade.pnl_net:+.0f}元  原因: {trade.exit_reason}',
        color=pnl_color, fontsize=11
    )
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(alpha=0.3)
    ax1.set_ylabel('價格（元）')
    ax2.bar(sub.index, sub['Volume'], color='#90A4AE', alpha=0.8)
    ax2.set_ylabel('成交股數')
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
%%time
# ── 修改這裡指定要分析哪幾筆交易（留空 = 自動取最大獲利/虧損各3筆）──
trades_to_analyze = [
    # ('2330', '2020-01-15'),
]
# ──────────────────────────────────────────────────────────────────

if not trades_to_analyze:
    top3_win  = df.nlargest(3, 'pnl_net')[['ticker','entry_date']].values.tolist()
    top3_loss = df.nsmallest(3, 'pnl_net')[['ticker','entry_date']].values.tolist()
    trades_to_analyze = [(str(int(t)), str(d.date())) for t, d in top3_win + top3_loss]

for ticker, entry_date in trades_to_analyze:
    plot_trade(ticker, entry_date, df)

## 10. 持倉天數分析

In [ ]:
%%time
import numpy as np

bins   = [0, 3, 7, 14, 30, 60, 120, 9999]
labels = ['1-3天','4-7天','8-14天','15-30天','31-60天','61-120天','120天+']
df['hold_bucket'] = pd.cut(df.hold_days, bins=bins, labels=labels)

grp = df.groupby('hold_bucket', observed=True).agg(
    筆數   =('pnl_net', 'count'),
    勝率   =('pnl_net', lambda x: f'{(x>0).mean():.1%}'),
    平均損益=('pnl_net', 'mean'),
    平均獲利=('pnl_net', lambda x: x[x>0].mean() if (x>0).any() else 0),
    平均虧損=('pnl_net', lambda x: x[x<0].mean() if (x<0).any() else 0),
    總損益  =('pnl_net', 'sum'),
)
grp['損益比'] = (grp['平均獲利'] / grp['平均虧損'].abs()).round(2).astype(str) + 'x'
print(f'[{LAYER}] 持倉天數分析')
print('='*70)
print(grp[['筆數','勝率','損益比','平均損益','總損益']].to_string())
print()
for bucket, g in df.groupby('hold_bucket', observed=True):
    wins  = g[g.pnl_net > 0].pnl_net
    loss  = g[g.pnl_net < 0].pnl_net
    total = g.pnl_net.sum()
    ratio = (wins.mean() / abs(loss.mean())) if (len(wins)>0 and len(loss)>0) else float('inf')
    flag  = ' ✓' if total > 0 else ' ✗'
    print(f"{str(bucket):10s}  {len(g):4d}筆  勝率={(g.pnl_net>0).mean():.1%}  損益比={ratio:.2f}x  總損益={total:+.0f}{flag}")

## 11. MDD 期間持倉診斷

In [ ]:
%%time
import numpy as np

eq        = pd.read_csv(EQUITY_CSV, index_col=0, parse_dates=True)['equity']
peak      = eq.cummax()
drawdown  = (eq - peak) / peak
mdd_val   = drawdown.min()
mdd_end   = drawdown.idxmin()
mdd_start = eq.loc[:mdd_end].idxmax()

print(f'[{LAYER}] MDD 診斷')
print('=' * 55)
print(f'  MDD 期間   : {mdd_start.date()} → {mdd_end.date()}')
print(f'  持續天數   : {(mdd_end - mdd_start).days} 天')
print(f'  最大回撤   : {mdd_val:.1%}')
print()

events = {
    ('2011-05-01', '2011-12-31'): '2011 歐債危機',
    ('2015-06-01', '2016-02-28'): '2015 中國股災 / 台股修正',
    ('2018-09-01', '2019-01-31'): '2018 美中貿易戰',
    ('2020-01-01', '2020-05-31'): '2020 COVID 急跌',
    ('2022-01-01', '2022-12-31'): '2022 升息熊市',
}
for (s, e), name in events.items():
    if max(mdd_start, pd.Timestamp(s)) <= min(mdd_end, pd.Timestamp(e)):
        print(f'  ✓ 與「{name}」重疊')
print()

active = df[(df['entry_date'] <= mdd_end) & (df['exit_date'] >= mdd_start)].copy()
print(f'  MDD 期間活躍交易：{len(active)} 筆')
if not active.empty:
    win_cnt = (active['pnl_net'] > 0).sum()
    print(f'  其中虧損：{len(active) - win_cnt} 筆（{(len(active)-win_cnt)/len(active):.0%}）')
    print(f'  合計損益：{active["pnl_net"].sum():,.0f} 元')
    print()
    cols = ['ticker','entry_date','exit_date','hold_days','pnl_net','exit_reason']
    print(active[cols].sort_values('pnl_net').to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 4))
eq.plot(ax=ax, color='steelblue', linewidth=1.2)
ax.axvspan(mdd_start, mdd_end, color='tomato', alpha=0.25, label=f'MDD ({mdd_val:.1%})')
ax.set_title(f'[{LAYER}] 淨值曲線 — MDD 期間標示', fontsize=13)
ax.set_ylabel('資產（元）')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print()
print('── 診斷結論 ──────────────────────────────────────')
if len(active) > 0:
    if (active['pnl_net'] < 0).mean() >= 0.7:
        print('  大部分持倉同時虧損 → 市場系統性下跌')
    else:
        print('  持倉損益不一致 → 少數大虧損，非整體市場問題')

## 12. 時間段績效分析

In [ ]:
%%time
import numpy as np

eq = pd.read_csv(EQUITY_CSV, index_col=0, parse_dates=True)['equity']

rolling_ret = eq.pct_change(252) * 100
fig, ax = plt.subplots(figsize=(14, 4))
rolling_ret.plot(ax=ax, color='#1565C0', linewidth=1.2, label='12M 滾動報酬率')
ax.axhline(0, color='#C62828', linewidth=0.8, linestyle='--')
ax.fill_between(rolling_ret.index, rolling_ret.where(rolling_ret >= 0, 0),
                color='#2E7D32', alpha=0.25, label='正報酬')
ax.fill_between(rolling_ret.index, rolling_ret.where(rolling_ret < 0, 0),
                color='#C62828', alpha=0.25, label='負報酬')
ax.set_title(f'[{LAYER}] 12 個月滾動報酬率', fontsize=13)
ax.set_ylabel('報酬率 (%)')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

monthly = eq.resample('ME').last().pct_change() * 100
monthly.index = monthly.index.to_period('M')
monthly_df = monthly.to_frame('ret')
monthly_df['year']  = monthly_df.index.year
monthly_df['month'] = monthly_df.index.month
pivot = monthly_df.pivot_table(index='year', columns='month', values='ret')
pivot.columns = ['1月','2月','3月','4月','5月','6月','7月','8月','9月','10月','11月','12月']

vmax = max(abs(pivot.values[~np.isnan(pivot.values)]).max(), 1)
fig, ax = plt.subplots(figsize=(16, len(pivot) * 0.6 + 1.5))
im = ax.imshow(pivot.values, cmap=plt.cm.RdYlGn, vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(12))
ax.set_xticklabels(pivot.columns, fontsize=10)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index, fontsize=10)
ax.set_title(f'[{LAYER}] 月度報酬熱力圖（%）', fontsize=13)
for i in range(len(pivot)):
    for j in range(12):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                    fontsize=8, color='black' if abs(val) < vmax * 0.6 else 'white')
plt.colorbar(im, ax=ax, label='月報酬率 (%)', shrink=0.8)
plt.tight_layout()
plt.show()

print(f'[{LAYER}] 各年度表現')
print('─' * 45)
annual = eq.resample('YE').last().pct_change() * 100
annual.index = annual.index.year
for yr, ret in annual.dropna().items():
    flag = '✓' if ret > 0 else '✗'
    bar  = '█' * int(abs(ret) / 2)
    print(f'  {yr}  {ret:+6.1f}%  {flag}  {bar}')
pos_years = (annual.dropna() > 0).sum()
neg_years = (annual.dropna() <= 0).sum()
print(f'\n  正報酬年份：{pos_years} 年  負報酬年份：{neg_years} 年')
print(f'  最佳：{annual.dropna().idxmax()}（{annual.dropna().max():+.1f}%）  '
      f'最差：{annual.dropna().idxmin()}（{annual.dropna().min():+.1f}%）')